In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from os import path 

import dotenv as de

from scipy.stats import pearsonr

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree

In [9]:
DATASET_NAME = de.get_key(".env", "DATASET_NAME")

data = pd.read_excel(f"data/{DATASET_NAME}", sheet_name = [1,2]) #Load in the 2nd and 3rd sheets of the excel file (all other sheets contain metadata and are not relevant)

In [10]:
DATA_ORIGINAL = pd.concat([data[1], data[2]], axis=0, ignore_index=True)

In [11]:
def preprocess_dataframe(df_orig):
    df = df_orig.copy() #Copy the original dataset

    df['Fan_Mailing_List'] = df['Fan_Mailing_List'].map({"Yes" : 1, "No" : 0}).astype(bool)
    df['Seat_Is_Upper'] = df['Seat_Location'].map({"Upper" : 1, "Lower" : 0}).astype(bool)

    df['Days_Before_Game'] = df['Days_Before_Game'].astype("float64")

    return df 

DATA = preprocess_dataframe(DATA_ORIGINAL)

In [12]:
#DATA.head()

## Model Building

#### Models to test:
- Decision Tree Regressor
- KNN
- Linear Regression (LASSO)

Try running Multi-Task Gradient Boosting Machine (from this paper: https://arxiv.org/abs/2201.06239)

Multi-Task Learning with Neural Networks: https://medium.com/data-scientists-diary/a-guide-to-multi-task-learning-in-machine-learning-768d22b88715

In [13]:
columns = ['Age', 'Fan_Mailing_List', 'Num_Tickets_Purchased', 'Seat_Is_Upper', 'Ticket_Price']
X = DATA[columns]
y = DATA['Days_Before_Game']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Experiment with different model families to get a base metric (to further optimize later)
models = [
    LinearRegression(), 
    DecisionTreeRegressor(max_depth=4, random_state=42, ccp_alpha=0.0), 
    RandomForestRegressor(n_estimators=100, random_state=42, ccp_alpha=0.0),
    Lasso(max_iter=10000)
]

for model in models:
    print(f"Model: {model}:")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    predictions = pd.DataFrame({"y_actuals" : y_test, "y_pred" : y_pred})
    print(predictions.head())
    print("R-squared Score:", r2_score(y_test, y_pred))
    print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
    print()

Model: LinearRegression():
     y_actuals     y_pred
926        9.0  11.659022
630       17.0  18.041986
682       26.0  21.645417
514       35.0  19.320800
365        0.0   1.096056
R-squared Score: 0.6335379635328342
Mean Absolute Error: 4.681822276975642

Model: DecisionTreeRegressor(max_depth=4, random_state=42):
     y_actuals     y_pred
926        9.0  10.171429
630       17.0  14.179487
682       26.0  23.015873
514       35.0  16.642857
365        0.0   3.685185
R-squared Score: 0.6142564973688549
Mean Absolute Error: 4.649128377914279

Model: RandomForestRegressor(random_state=42):
     y_actuals  y_pred
926        9.0   10.35
630       17.0   13.02
682       26.0   23.01
514       35.0   14.43
365        0.0    2.29
R-squared Score: 0.6474244234676463
Mean Absolute Error: 4.345074626865671

Model: Lasso(max_iter=10000):
     y_actuals     y_pred
926        9.0   9.903022
630       17.0  16.643415
682       26.0  21.753168
514       35.0  17.124584
365        0.0  -1.868020
R-